In [1]:
import json
import glob
import torch
import pyspark
import findspark
import torch.nn as nn
from pyspark.sql import SparkSession
from sklearn.model_selection import train_test_split

In [2]:
# spark = SparkSession.builder.appName("JsonExtractor").getOrCreate()

findspark.init()

spark = SparkSession.builder \
    .appName("JsonExtractor") \
    .config("spark.driver.port", "5000")\
    .config("spark.blockManager.port", "4000")\
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/10 20:07:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
dir = "archive/batch_*/json/kaggle_data_*.json"

json_files = glob.glob(dir)

sorted(json_files)

['archive/batch_1/json/kaggle_data_1.json',
 'archive/batch_10/json/kaggle_data_10.json',
 'archive/batch_2/json/kaggle_data_2.json',
 'archive/batch_3/json/kaggle_data_3.json',
 'archive/batch_4/json/kaggle_data_4.json',
 'archive/batch_5/json/kaggle_data_5.json',
 'archive/batch_6/json/kaggle_data_6.json',
 'archive/batch_7/json/kaggle_data_7.json',
 'archive/batch_8/json/kaggle_data_8.json',
 'archive/batch_9/json/kaggle_data_9.json']

In [4]:
with open(json_files[1]) as file:
    data = json.load(file)

# print(data)

In [5]:
image_data = spark.read.json(json_files[0])

In [12]:
image_data.show()

+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|            filename|   font|          image_data|               latex|unicode_less_curlies|         unicode_str|                uuid|
+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|bd85ee85-2549-453...|Oct_011|{4, [\lim_, {, a,...|\lim_{a\to\frac{\...|ƀaƄಭೃ4ಭಭddaಷƇa+-6...|ƀ{aƄಭ{ೃ}{4}}ಭ{ಭ{d...|bd85ee85-2549-453...|
|e97b0b1f-08bf-4c2...|   C017|{4, [\lim_, {, w,...|\lim_{w\to\pi/5^{...|ƀwƄೃ/5^-ಭƉ^2w2ಭ-4...|ƀ{wƄೃ/5^{-}}ಭ{Ɖ^{...|e97b0b1f-08bf-4c2...|
|3c72e1a1-c1de-4d6...|Oct_095|{4, [e, ^, {, \li...|e^{\lim_{b\to4^{+...|    e^ƀbƄ4^+ಭಭ7b+3ƅb|e^{ƀ{bƄ4^{+}}ಭ{ಭ{...|3c72e1a1-c1de-4d6...|
|54459370-7cbe-423...|Oct_203|{4, [\lim_, {, \t...|\lim_{\theta\to\p...|ƀ೨Ƅೃ/8Ɖ^3೨+ƀ೨Ƅೃ/6...|ƀ{೨Ƅೃ/8}Ɖ^{3}{೨}+...|54459370-7cbe-423...|
|da71ce9d-e1a1-4bb...|   C004|{4, [=, \lim_, {,.

In [13]:
from pyspark.sql.functions import StructType, explode, col
from pyspark.sql.types import IntegerType, FloatType, DoubleType, StringType, StructType, StructField

image_prop = spark.createDataFrame([],
                schema = StructType([
                    StructField('full_latex_chars', StringType()),
                    StructField('visible_latex_chars', StringType()),
                    StructField('visible_char_map', StringType()),
                    StructField('width', FloatType()),
                    StructField('height', FloatType()),
                    StructField('depth', FloatType()),
                    StructField('xmins', DoubleType()),
                    StructField('ymins', DoubleType()),
                    StructField('xmaxs', DoubleType()),
                    StructField('ymaxs', DoubleType()),
                    StructField('xmins_raw', IntegerType()),
                    StructField('ymins_raw', IntegerType()),
                    StructField('xmaxs_raw', IntegerType()),
                    StructField('ymaxs_raw', IntegerType()),
                ])
            )

In [ ]:
image_label = image_data.select(explode('image_data.full_latex_chars').alias('image'))

In [16]:
image_label.show()

+-----+
|image|
+-----+
|\lim_|
|    {|
|    a|
|  \to|
|\frac|
|    {|
|  \pi|
|    }|
|    {|
|    4|
|    }|
|    }|
|\frac|
|    {|
|\frac|
|    {|
|    d|
|    }|
|    {|
|    d|
+-----+
only showing top 20 rows



In [19]:
image_label_array = image_data.select('image_data.full_latex_chars')

image_label_array.show(truncate = False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|full_latex_chars                                                                                                                                                                                                                                       |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[\lim_, {, a, \to, \frac, {, \pi, }, {, 4, }, }, \frac, {, \frac, {, d, }, {, d, a, }, \left(, \sin, {, a, }, +, -, 6, \sec, {, a, }, \right), }, {, \frac, {, d, }, {, d, a, }, \left(, a, +, -, 4, \frac, {, \pi, }, {, 4, }, \right), }]            |


In [29]:
image_data = image_data.\
                withColumn('full_latex_chars', col('image_data.full_latex_chars')).\
                withColumn('visible_latex_chars', col('image_data.visible_latex_chars')).\
                withColumn('visible_char_map', col('image_data.visible_char_map')).\
                withColumn('width', col('image_data.width')).\
                withColumn('height', col('image_data.height')).\
                withColumn('depth', col('image_data.depth')).\
                withColumn('xmins', col('image_data.xmins')).\
                withColumn('xmaxs', col('image_data.xmaxs')).\
                withColumn('ymins', col('image_data.ymins')).\
                withColumn('ymaxs', col('image_data.ymaxs')).\
                withColumn('xmins_raw', col('image_data.xmins_raw')).\
                withColumn('xmaxs_raw', col('image_data.xmaxs_raw')).\
                withColumn('ymins_raw', col('image_data.ymins_raw')).\
                withColumn('ymaxs_raw', col('image_data.ymaxs_raw')).\
                withColumn('png_masks', col('image_data.png_masks'))

image_data.show()

+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|            filename|   font|          image_data|               latex|unicode_less_curlies|         unicode_str|                uuid|    full_latex_chars| visible_latex_chars|    visible_char_map|width|height|depth|               xmins|               xmaxs|               ymins|               ymaxs|           xmins_raw|           xmaxs_raw|           ymins_raw|           ymaxs_raw|           png_masks|
+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------

In [ ]:
## A better way to organise the table

# from pyspark.sql.functions import col, concat_ws
# from pyspark.ml.feature import VectorAssembler

# # Example: If you want to convert an array of LaTeX tokens into a single string (if that makes sense for your use case)
# df = df.withColumn("full_latex_string", concat_ws(" ", col("full_latex_chars")))

# # Let's assume you also want to use numeric columns like width, height, and depth as features.
# feature_cols = ["width", "height", "depth"]  # add any other numeric features you need

# # Assemble into a feature vector
# assembler = VectorAssembler(inputCols = feature_cols, outputCol = "features")
# df_transformed = assembler.transform(df)

# # Now df_transformed has a 'features' column that you can use for model training.
# df_transformed.select("features", "latex").show(truncate = False)

In [33]:
# from pyspark.sql.functions import explode, col
# from PIL import Image
# import io

# def decode_png_mask(png_data):
#     """Decodes a PNG string into a Pillow Image object."""
#     try:
#         image = Image.open(io.BytesIO(png_data))
#         return image
#     except Exception as e:
#         # Handle errors (e.g., invalid PNG data)
#         return None

# # Assuming 'df' is your Spark DataFrame
# exploded_df = df.select(
#     "*", 
#     explode(col("image_data.png_masks")).alias("png_mask_data")  # Explode png_masks
# )

# # Decode PNG data to Pillow Image
# decoded_df = exploded_df.rdd.map(
#     lambda row: row + (decode_png_mask(row.png_mask_data),)
# ).toDF(["filename", "font", "image_data", "latex", "unicode_less_curlies", "unicode_str", "uuid", "png_mask_data", "png_mask_image"])

In [ ]:
import re

def clearDigits(self, s: str) -> str:

    r_string = r"\w\d"

    s2 = re.sub(r_string, "", str01)

    return s2

str01 = "eg5th5h4"



str02

'et'